<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/Ai/blob/main/Basic_text_classifier_from_image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [107]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np

Data Processing

In [108]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

Loading MNIST data


In [109]:
train_dataset = torchvision.datasets.MNIST(
    root="./data" , train = True , download= True , transform= transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data" , train = False , download= True , transform= transform
)

Data loader

In [110]:
train_loader = DataLoader(train_dataset , batch_size=64 , shuffle=True)
test_loader = DataLoader(test_dataset , batch_size=1000 , shuffle=False)


creating Neural Network


In [111]:
class MNISTClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.layers = nn.Sequential(
        nn.Linear(784 , 128),
        nn.ReLU(),
        nn.Linear(128 ,10)
    )

  def forward(self , X):
    x = self.flatten(X)
    x = self.layers(x)
    return x


In [112]:
from torch.optim.optimizer import Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MNISTClassifier().to(device)

loss_fn = nn.CrossEntropyLoss()
Optimizer = optim.Adam(model.parameters() , lr=0.001)

Training

In [113]:
def train_epoch(model , train_loader , loss_fn , Optimizer , device):
  model.train()
  running_loss = 0
  correct = 0
  total = 0

  for batch_idx , (data , target) in enumerate(train_loader):
    data , target = data.to(device) , target.to(device)

    Optimizer.zero_grad()
    output = model(data)
    loss = loss_fn(output , target)
    loss.backward()
    Optimizer.step()

    running_loss += loss
    _, predict = output.max(1)
    total += target.size(0)
    correct += predict.eq(target).sum().item()

    if batch_idx % 100 == 0 and batch_idx > 0 :
      avg_loss = running_loss/100
      accuracy = 100 * correct / total
      print(f'[{batch_idx * 64}/{60000}]'
            f'Loss: {avg_loss: .3f} |  Accuracy: {accuracy: .1f}%')
      running_loss = 0

In [116]:
def evalution(model , test_loader , device):
  model.eval()
  correct = 0
  total = 0

  with torch.no_grad():
    for input , target in test_loader:
      input , target = input.to(device) , target.to(device)
      output = model(input)
      _ , predict = output.max(1)
      total += target.size(0)
      correct += predict.eq(target).sum().item()
  return 100 * correct/total

In [117]:
num_epoch = 10
for epoch in range(num_epoch):
  print(f'\nEpoch: {epoch+1}')
  train_epoch(model , train_loader , loss_fn , Optimizer , device)
  accuracy = evalution(model ,test_loader , device)
  print("Test Accuracy: " , accuracy)


Epoch: 1
[6400/60000]Loss:  0.138 |  Accuracy:  95.9%
[12800/60000]Loss:  0.125 |  Accuracy:  96.1%
[19200/60000]Loss:  0.123 |  Accuracy:  96.2%
[25600/60000]Loss:  0.114 |  Accuracy:  96.3%
[32000/60000]Loss:  0.115 |  Accuracy:  96.3%
[38400/60000]Loss:  0.113 |  Accuracy:  96.4%
[44800/60000]Loss:  0.112 |  Accuracy:  96.4%
[51200/60000]Loss:  0.112 |  Accuracy:  96.4%
[57600/60000]Loss:  0.100 |  Accuracy:  96.5%
Test Accuracy:  97.18

Epoch: 2
[6400/60000]Loss:  0.073 |  Accuracy:  97.9%
[12800/60000]Loss:  0.083 |  Accuracy:  97.6%
[19200/60000]Loss:  0.078 |  Accuracy:  97.7%
[25600/60000]Loss:  0.090 |  Accuracy:  97.6%
[32000/60000]Loss:  0.090 |  Accuracy:  97.5%
[38400/60000]Loss:  0.077 |  Accuracy:  97.5%
[44800/60000]Loss:  0.088 |  Accuracy:  97.5%
[51200/60000]Loss:  0.079 |  Accuracy:  97.5%
[57600/60000]Loss:  0.070 |  Accuracy:  97.5%
Test Accuracy:  96.9

Epoch: 3
[6400/60000]Loss:  0.057 |  Accuracy:  98.3%
[12800/60000]Loss:  0.053 |  Accuracy:  98.4%
[19200/600